# 01 · Ingesta a bronze
**Responsable:** P6 · **Capa:** staging → bronze · **Código:** `src/ingesta_bronze.py`, `src/calidad.py`, `src/mocks.py`

**Qué hace**
1. Borra las métricas de la corrida anterior (cada corrida deja su propio registro).
2. Copia los 4 CSV de `staging/` a `data_lake/bronze/<fuente>/` **sin modificarlos**.
3. Convierte los 2 GeoJSON a CSV con la geometría en una columna WKT (regla de la sección 4 del enunciado).
4. Escribe un manifiesto por fuente: URL, entidad, hash SHA-256 del original y del CSV, tamaño, codificación, filas, columnas, CRS y pasos de conversión.
5. Genera los mocks de `tests/fixtures/` para quien quiera programar sin datos reales.

**Qué revisar al ejecutar:** que las 6 fuentes digan `[ok]` y que `manifest_bronze.json` no tenga errores. Completar a mano la `fecha_descarga` en cada `manifest.json`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import polars as pl
from src.config import *
asegurar_carpetas()

In [2]:
from src.calidad import reiniciar_metricas
from src.ingesta_bronze import main as ingerir
reiniciar_metricas()
ingerir()

[ok] evaluaciones_agropecuarias: 2613 filas, 20 columnas
[ok] centros_poblados: 1845 filas, 266 columnas
[ok] desempeno_municipal: 246 filas, 15 columnas
[ok] poblacion_dane: 738 filas, 10 columnas
[ok] municipios_mgn: 123 filas, 4 columnas
[ok] capacidad_uso_igac: 958 filas, 15 columnas


In [3]:
import json
m = json.loads((BRONZE / "manifest_bronze.json").read_text(encoding="utf-8"))
pl.DataFrame([{k: f[k] for k in ("fuente", "filas", "columnas", "codificacion", "crs", "filas_duplicadas_exactas")}
              for f in m["fuentes"]])

fuente,filas,columnas,codificacion,crs,filas_duplicadas_exactas
str,i64,i64,str,str,i64
"""evaluaciones_agropecuarias""",2613,20,"""utf-8""",null,0
"""centros_poblados""",1845,266,"""utf-8""",null,0
"""desempeno_municipal""",246,15,"""utf-8""",null,0
"""poblacion_dane""",738,10,"""utf-8""",null,0
"""municipios_mgn""",123,4,"""utf-8""","""EPSG:4326""",0
"""capacidad_uso_igac""",958,15,"""utf-8""","""EPSG:4326""",0


In [4]:
from src.mocks import generar_mocks
generar_mocks()

Mocks escritos en /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/proyecto_boyaca/tests/fixtures
